# Taller 4 - Analisis exploratorio de Disney API

Este notebook lee desde MongoDB los datos crudos guardados por `ingesta.py`, selecciona variables relevantes y realiza un EDA con cinco insights y tres graficos.

# 1. Importar librerias y conectar con MongoDB

In [4]:
from pymongo import MongoClient
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "taller4_db"
COLLECTION_NAME = "raw_data"

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

total_documentos = collection.count_documents({})
print(f"Documentos encontrados en MongoDB: {total_documentos}")

Documentos encontrados en MongoDB: 100


# 2. Cargar datos crudos y seleccionar variables

Lectura de documentos crudos desde MongoDB y construcción de DataFrame con variables utiles para el analisis.

In [5]:
raw_data = list(collection.find())
df_raw = pd.DataFrame(raw_data)
df_raw.head()

,_id,films,shortFilms,tvShows,videoGames,parkAttractions,allies,enemies,name,imageUrl,url,alignment
0,112,[Hercules (film)],[],[Hercules (TV series)],[Kingdom Hearts III],[],[],[],Achilles,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/112,NaN
1,18,"[The Fox and the Hound, The Fox and the Hound 2]",[],[],[],[],[],[],Abigail the Cow,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/18,NaN
2,16,[Cheetah],[],[],[],[],[],[],Abdullah,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/16,NaN
3,45,"[Mary Poppins (film), Mary Poppins Returns]",[],[],[],[Disney Movie Magic],[],[],Admiral Boom and Mr. Binnacle,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/45,NaN
4,7,[],[],[Gravity Falls],[Disney Heroes: Battle Mode],[],[],[],.GIFfany,https://static.wikia.nocookie.net/disney/image...,https://api.disneyapi.dev/characters/7,NaN


In [6]:
def contar_lista(valor):
    return len(valor) if isinstance(valor, list) else 0

df = pd.DataFrame({
    "name": df_raw["name"],
    "films_count": df_raw["films"].apply(contar_lista),
    "short_films_count": df_raw["shortFilms"].apply(contar_lista),
    "tv_shows_count": df_raw["tvShows"].apply(contar_lista),
    "video_games_count": df_raw["videoGames"].apply(contar_lista),
    "park_attractions_count": df_raw["parkAttractions"].apply(contar_lista),
    "allies_count": df_raw["allies"].apply(contar_lista),
    "enemies_count": df_raw["enemies"].apply(contar_lista),
    "has_image": df_raw["imageUrl"].notna()
})

columnas_apariciones = ["films_count", "short_films_count", "tv_shows_count", "video_games_count", "park_attractions_count"]
df["total_appearances"] = df[columnas_apariciones].sum(axis=1)
df["has_films"] = df["films_count"].apply(lambda x: "Con peliculas" if x > 0 else "Sin peliculas")

df.head()

,name,films_count,short_films_count,tv_shows_count,video_games_count,park_attractions_count,allies_count,enemies_count,has_image,total_appearances,has_films
0,Achilles,1,0,1,1,0,0,0,True,3,Con peliculas
1,Abigail the Cow,2,0,0,0,0,0,0,True,2,Con peliculas
2,Abdullah,1,0,0,0,0,0,0,True,1,Con peliculas
3,Admiral Boom and Mr. Binnacle,2,0,0,0,1,0,0,True,3,Con peliculas
4,.GIFfany,0,0,1,1,0,0,0,True,2,Sin peliculas
